# 🏛️ Crypto Institutional Quant Fund - Cloud Training Hub (Google Colab)
### Dual-Engine AI Trading System: LightGBM + PyTorch Time-Series Transformer
---
هذه المفكرة مخصصة لتدريب واختبار النماذج العميقة والهجينة على سحابة Google Colab مستفيدة من كروت الشاشة المجانية (NVIDIA T4 GPU) وسرعة المعالجة الفائقة، وتصدير النموذج المدرب مباشرة إلى جهازك المحلي دون استهلاك موارده.

## 1️⃣ التحقق من كرت الشاشة (GPU Verification)
تأكد أولاً أن بيئة التشغيل مفعلة على GPU من القائمة العليا:
`Runtime` -> `Change runtime type` -> اختر **T4 GPU**.

In [ ]:
!nvidia-smi

## 2️⃣ ربط Google Drive لحفظ النماذج تلقائياً

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive Connected!')

## 3️⃣ استنساخ مستودع المشروع (Private Repo)
قم بوضع رمز GitHub Personal Access Token (PAT) لاستنساخ المستودع الخاص وفرع التطوير المعزول.

In [ ]:
import os
# استبدل your_github_token بالتوكن الخاص بك من GitHub (Settings -> Developer Settings -> Personal access tokens)
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/EmranYaseen-ALSabry/Crypto-Institutional-Quant-Fund.git"

%cd /content
if not os.path.exists('/content/Crypto-Institutional-Quant-Fund'):
    !git clone -b feature/hybrid-transformer $REPO_URL
else:
    %cd /content/Crypto-Institutional-Quant-Fund
    !git pull origin feature/hybrid-transformer

%cd /content/Crypto-Institutional-Quant-Fund
print('✅ Cloned feature/hybrid-transformer branch successfully!')

## 4️⃣ تثبيت Freqtrade والمكتبات الرياضية مع دعم PyTorch & LightGBM

In [ ]:
# تثبيت المتطلبات الضرورية لنظام FreqAI والترانسفورمر
!pip install -q uv
!uv pip install --system freqtrade[freqai,pytorch] lightgbm joblib
print('✅ Environment & Dependencies Installed!')

## 5️⃣ تحميل وتحديث بيانات الشموع (1h & 4h) من Binance

In [ ]:
!freqtrade download-data \
    --config user_data/config_spot_hybrid.json \
    --timeframes 1h 4h \
    --timerange 20260101-20261105 \
    --exchange binance

## 6️⃣ تدريب النموذج الهجين والباك تيست (Walk-Forward on GPU T4) 🚀
يقوم هذا الأمر بتدريب شبكة الترانسفورمر المالي ونموذج LightGBM معاً باستخدام كرت الشاشة.

In [ ]:
!freqtrade backtesting \
    --config user_data/config_spot_hybrid.json \
    --strategy InstitutionalDualStrategy \
    --freqaimodel HybridTreeTransformer \
    --timerange 20260901-20261105

## 7️⃣ تصدير النموذج المدرب مباشرة إلى Google Drive
يتم ضغط مجلد أوزان النموذج الجديد `spot_hybrid_transformer_model` وحفظه في حسابك.

In [ ]:
import shutil
from pathlib import Path

model_dir = Path('user_data/models/spot_hybrid_transformer_model')
zip_dest = '/content/drive/MyDrive/spot_hybrid_transformer_model'

if model_dir.exists():
    print('📦 Zipping model weights...')
    shutil.make_archive(zip_dest, 'zip', model_dir)
    print(f'🎉 Success! Model saved to Google Drive at: {zip_dest}.zip')
    print('👉 You can now download this zip and extract it into your local user_data/models/ folder!')
else:
    print('⚠️ Model directory not found. Please run Step 6 first.')